# Gaia DR4 EAS + radial velocity orbit fit: HD 114762

This notebook first fits the published radial velocities of **HD 114762**, then adds the Gaia DR4 prerelease epoch astrometry for a joint fit. The RV posterior supplies physically informed starting positions for the joint sampler.

**Gaia data credit: ESA/Gaia/DPAC.** See the [DR4 prerelease](https://www.cosmos.esa.int/web/gaia/dr4-prerelease) and [data provenance](../../orbitize/example_data/gaia_dr4_eas_README.md).

The 74 Lick/Hamilton radial velocities are from [Kane et al. (2011)](https://doi.org/10.1088/2041-8205/735/2/L41).

In [ ]:
from pathlib import Path
from time import perf_counter
import sys
from astropy.time import Time

repo_root = Path("../..").resolve()
sys.path.insert(0, str(repo_root))

import corner
import numpy as np
import matplotlib.pyplot as plt
from astropy import units as u
import orbitize
from orbitize import gaia, priors, read_input, sampler, system

rng = np.random.default_rng(42)
source_id = 3937211745905473024

# from Exoplanet Archive https://exoplanetarchive.ipac.caltech.edu/overview/HD%20114762
stellar_mass = 0.83       # Msun
stellar_mass_err = 0.01   # Msun
ref_epoch_mjd = 57936.375 # J2017.5, MJD(TCB)
dr3_plx =24.8556
dr3_plx_err =0.2743

## 1. Fit the radial velocities

Kane et al. obtained these measurements with the Hamilton spectrograph over 19 years. Detector changes introduce independent RV zero points, so each dewar is retained as a separate orbitize instrument.

In [ ]:
# GET RV DATA
rv_path = Path(orbitize.DATADIR) / "hd_114762_rv_kane2011.csv"
rv_data = read_input.read_file(rv_path)

print(f"{len(rv_data)} RV measurements")
print("Instruments:", ", ".join(np.unique(rv_data["instrument"])))

fig, ax = plt.subplots(figsize=(10, 4))
for instrument in np.unique(rv_data["instrument"]):
    use = rv_data["instrument"] == instrument
    ax.errorbar(
        rv_data["epoch"][use],
        1_000 * rv_data["quant1"][use],
        yerr=1_000 * rv_data["quant1_err"][use],
        fmt=".", label=instrument,
    )
ax.set(xlabel="MJD", ylabel="Relative RV [m/s]")
ax.legend(fontsize=8, ncol=3)
fig.tight_layout()

In [ ]:
# CREATE SYSTEM FOR RV FIT

rv_system = system.System(
    1,
    rv_data,
    stellar_mass,
    dr3_plx,
    mass_err=stellar_mass_err,
    plx_err=dr3_plx_err,
    fit_secondary_mass=True,
    tau_ref_epoch=ref_epoch_mjd,
    fitting_basis="SemiAmp",
)
rv_idx = rv_system.param_idx

# SemiAmp samples period in years and stellar RV semi-amplitude in km/s.
rv_system.sys_priors[rv_idx["per1"]] = priors.UniformPrior(0.15, 0.5)
rv_system.sys_priors[rv_idx["ecc1"]] = priors.UniformPrior(0.0, 0.99)
rv_system.sys_priors[rv_idx["K1"]] = priors.LogUniformPrior(0.1, 1.5)
rv_system.sys_priors[rv_idx["inc1"]] = priors.SinPrior()
rv_system.sys_priors[rv_idx["pan1"]] = priors.UniformPrior(0, 2 * np.pi)

for instrument in rv_system.rv_instruments:
    rv_system.sys_priors[rv_idx[f"gamma_{instrument}"]] = (
        priors.UniformPrior(-1, 1)
    )
    rv_system.sys_priors[rv_idx[f"sigma_{instrument}"]] = (
        priors.LogUniformPrior(1e-4, 0.1)
    )

In [ ]:
# SET UP SAMPLER

rv_num_temps = 10
rv_num_walkers = 128
rv_burn_steps = 100
rv_num_steps = 500
# for convergence
# rv_burn_steps = 2_000
# rv_num_steps = 5_000
thin=1
rv_mcmc = sampler.MCMC(
    rv_system,
    num_temps=rv_num_temps,
    num_walkers=rv_num_walkers,
    num_threads=10,
)
rv_run = rv_mcmc.run_sampler(
    rv_num_walkers * rv_num_steps,
    burn_steps=rv_burn_steps,
        thin=thin,
)
rv_post = rv_mcmc.results.post

print(
    f"{rv_num_steps} production steps; "
    f"cold-chain acceptance={np.mean(rv_run.acceptance_fraction[0]):.3f}"
)

In [ ]:
rv_corner = rv_mcmc.results.plot_corner(
    param_list=["per1", "ecc1", "K1", "aop1", "tau1"],
    labels=["P [yr]", "e", "K [km/s]", r"$\omega$ [deg]", r"$\tau$"],
    quantiles=[0.16, 0.5, 0.84],
    show_titles=True,
    plot_datapoints=False,
    fill_contours=True,
)

# Published RV solution from Kane et al. (2011). Orbitize's
# aop1 describes the companion, so it is 180 deg from the published
# stellar argument of periastron. Convert T_peri to tau at J2017.5.
published_rv_fit = [
    83.9151 / 365.25,
    0.3354,
    0.61248,
    (201.28 - 180.0) % 360.0,
    ((2449889.106 - 2400000.5 - ref_epoch_mjd) / 83.9151) % 1,
]
corner.overplot_lines(
    rv_corner, published_rv_fit, color="tab:orange",
    linestyle="--", linewidth=1.5, label="Kane et al. (2011)",
)
rv_corner.legend(
    *rv_corner.axes[0].get_legend_handles_labels(),
    loc="upper right", frameon=False,
)

### Download the Gaia prerelease data

The EAS data are downloaded separately.

1. Download the [ESA epoch-astrometry ZIP](https://anonftp.cosmos.esa.int/pub/GAIA_PUBLIC_DATA/Gaia_DR4/dr4-prerelease/gaia-dr4-prerelease-epoch-astrometry_2026-06-26.zip).
2. Extract the archive.
3. Create a `downloads` folder beside this notebook and place
   `GAIA_DR4_PRERELEASE_EPOCH_ASTROMETRY_RAW.xml` inside it.

Data credit: ESA/Gaia/DPAC. See the
[prerelease information](https://www.cosmos.esa.int/web/gaia/dr4-prerelease)
and follow the acknowledgement instructions supplied in the archive's `readme.txt`.

This tutorial uses the Gaia DR4 prerelease XML bundled with orbitize!, so you don't need to complete this step.

In [ ]:
xml_path = Path(orbitize.DATADIR) / "GAIA_DR4_PRERELEASE_EPOCH_ASTROMETRY_RAW.xml"
data = read_input.from_dl2(xml_path, source_id)

In [ ]:
ccd_data = data
#data = read_input.combine_transits(ccd_data, n_boot=500, rng=rng)

print(f"Reduced {len(ccd_data)} CCD measurements to {len(data)} transits")
data[:5]

### Along-scan geometry

Time offsets and scan angles project position and proper motion onto the along-scan direction. During the orientation search, we solve those four linear terms after subtracting each trial orbit and parallax. They remain sampled in the joint MCMC fit.


In [ ]:
time_mjd = np.asarray(data["obs_time_tcb"], dtype=float)
al = np.asarray(data["centroid_pos_al"], dtype=float)
al_err = np.asarray(data["centroid_pos_error_al"], dtype=float)
psi = np.asarray(data["scan_pos_angle"], dtype=float)
parallax_factor = np.asarray(data["parallax_factor_al"], dtype=float)
dt_yr = (time_mjd - ref_epoch_mjd) / 365.25

sinpsi = np.sin(psi)
cospsi = np.cos(psi)

## 3. Joint RV + Gaia DR4 fit

The same 74 RVs now enter a joint likelihood with the 63 reduced Gaia transits. Gaia constrains the sky-plane orientation and true companion mass; the RVs retain the precise spectroscopic orbit.

The joint fit uses `SemiAmp`: period and RV amplitude are sampled, and companion mass and semimajor axis are derived at every likelihood evaluation. Period is uniform over 0.15–0.35 years and RV amplitude is log-uniform over 0.1–1.5 km/s, as in the RV-only fit. These replace the earlier semimajor-axis and mass priors; they are not an equivalent transformation of those priors. Inclination and node remain broad, with no published orientation used for initialization.

For initialization only, we anchor the spectroscopic orbit to the best RV-only sample, hold host mass and parallax at their prior centres, and search broadly over inclination, node, and Gaia jitter. Each candidate gets a conditional position/proper-motion solve. Walkers start close to the best candidate; all fitted parameters are then free again in MCMC. No published inclination or node is used. This locates a promising mode, not proof of global convergence.

The joint sampler runs once with no old chain history attached. `num_steps` below controls the production length; 1,000 steps is a short initial run, not a convergence criterion.


In [ ]:
dr4 = gaia.DR4LogProb(
    source_id,
    data,
    ref_epoch_mjd=ref_epoch_mjd,
    jitter_prior=priors.LogUniformPrior(1e-4, 100),
)

my_sys = system.System(
    1,
    rv_data,
    stellar_mass,
    dr3_plx,
    mass_err=stellar_mass_err,
    plx_err=dr3_plx_err,
    fit_secondary_mass=True,
    gaia=dr4,
    tau_ref_epoch=ref_epoch_mjd,
    fitting_basis="SemiAmp",
)
idx = my_sys.param_idx

for name, prior in {
    "per1": priors.UniformPrior(0.15, 0.35),
    "K1": priors.LogUniformPrior(0.1, 1.5),
    "ecc1": priors.UniformPrior(0, 0.99),
    "inc1": priors.SinPrior(),
    "aop1": priors.UniformPrior(0, 2 * np.pi),
    "pan1": priors.UniformPrior(0, 2 * np.pi),
    "tau1": priors.UniformPrior(0, 1),
    "alpha0": priors.UniformPrior(-100, 100),
    "delta0": priors.UniformPrior(-100, 100),
    "pm_ra": priors.UniformPrior(-2000, 2000),
    "pm_dec": priors.UniformPrior(-2000, 2000),
}.items():
    my_sys.sys_priors[idx[name]] = prior

for instrument in my_sys.rv_instruments:
    my_sys.sys_priors[idx[f"gamma_{instrument}"]] = priors.UniformPrior(-1, 1)
    my_sys.sys_priors[idx[f"sigma_{instrument}"]] = (
        priors.LogUniformPrior(1e-4, 0.1)
    )

In [ ]:
print(len(data), my_sys.gaia.n_obs)  # should print 63 63

In [ ]:
num_temps = 10
num_walkers = 128
num_threads = 10
# for convergence
# burn_steps = 2_000
# num_steps = 5_000
burn_steps = 100
num_steps = 500
thin = 1

assert rv_system.fitting_basis == my_sys.fitting_basis == "SemiAmp"

mcmc = sampler.MCMC(
    my_sys,
    num_temps=num_temps,
    num_walkers=num_walkers,
    num_threads=num_threads,
)

idx = my_sys.param_idx
rv_idx = rv_system.param_idx
sampled_idx = mcmc.sampled_param_idx
positions = mcmc.curr_pos.reshape(-1, mcmc.num_params)

# Copy random RV posterior draws, but keep inclination and LAN broad.
warm_names = [
    name for name in sampled_idx
    if name in rv_idx and name not in ("inc1", "pan1")
]
rv_columns = [rv_idx[name] for name in warm_names]
joint_columns = [sampled_idx[name] for name in warm_names]

# Use RV starting candidates supported by the joint period prior.
period_prior = mcmc.priors[mcmc.sampled_param_idx["per1"]]
rv_periods = np.asarray(rv_post)[:, rv_idx["per1"]]

valid = (
    np.isfinite(rv_periods)
    & np.isfinite(period_prior.compute_lnprob(rv_periods))
)
rv_start_post = np.asarray(rv_post)[valid]

if len(rv_start_post) == 0:
    raise ValueError("No RV starting candidates satisfy the joint period prior.")

draws = rng.choice(len(rv_start_post), size=len(positions), replace=True)
positions[:, joint_columns] = rv_start_post[draws][:, rv_columns]
# Inclination, LAN and Gaia jitter retain their joint prior draws.
full_positions = mcmc._fill_in_fixed_params(positions)
standard_positions = my_sys.basis.to_standard_basis(
    full_positions.T.copy()
)
ra, dec, _, _ = my_sys.compute_all_orbits(
    standard_positions, epochs=dr4.epochs_mjd
)
orbit_al = (
    ra[:, 0, :] * dr4.sin_scan[:, None]
    + dec[:, 0, :] * dr4.cos_scan[:, None]
)

# Solve position and proper motion only to initialize each walker.
linear_columns = [
    sampled_idx[name]
    for name in ("alpha0", "delta0", "pm_ra", "pm_dec")
]
linear_epochs = Time(
    dr4.epochs_mjd, format="mjd", scale=dr4.time_scale,
)
linear_ref = Time(
    dr4.ref_epoch_mjd, format="mjd", scale=dr4.time_scale,
).jyear
for walker, params in enumerate(full_positions):
    sigma = np.hypot(al_err, params[idx["dr4_jitter"]])
    target = (
        al - params[idx["plx"]] * parallax_factor
        - orbit_al[:, walker]
    )
    positions[walker, linear_columns] = gaia.HGCALogProb._linear_pm_fit(
        epochs=linear_epochs,
        raoff_planet=target * sinpsi,
        decoff_planet=target * cospsi,
        epoch_ref_ra=linear_ref,
        epoch_ref_dec=linear_ref,
        sin_phi=cospsi,
        cos_phi=sinpsi,
        errs=sigma,
    )

mcmc.check_prior_support()

In [ ]:
start = perf_counter()
print("Workers:", mcmc.num_threads)
run = mcmc.run_sampler(
    num_walkers * num_steps,
    burn_steps=burn_steps,
    thin=thin,
)
post = mcmc.results.post

print(f"Joint run took {(perf_counter() - start) / 60:.1f} minutes")
print(f"Cold-chain acceptance: {run.acceptance_fraction[0].mean():.3f}")
print(f"Best log-likelihood: {np.nanmax(mcmc.results.lnlike):.3f}")

y## 4. Inspect the joint posterior

In [ ]:
# check chain convergence

chain_params = [
    "per1", "ecc1", "inc1", "pan1", "K1", "plx", "dr4_jitter"
]
chain_figures = mcmc.examine_chains(
    param_list=chain_params,
    walker_list=range(16),
    transparency=0.35,
)

for fig, name in zip(chain_figures, chain_params):
    fig.axes[0].set_ylabel(name)

In [ ]:
import importlib
import orbitize.plot
importlib.reload(orbitize.plot)

with plt.style.context("default"):
    fig = mcmc.results.plot_dr4_orbit(
        mcmc.system,
        plot_map=True,
        num_epochs_to_plot=5_000,
        include_jitter=True,
        fontsize=14,
    )
    fig.axes[0].set_title("HD 114762: MAP joint RV + Gaia orbit")

In [ ]:
print("Basis:", mcmc.system.fitting_basis)
print("Temperatures:", np.round(1 / run.betas, 2))
print("Mean acceptance per temperature:",
      np.round(run.acceptance_fraction.mean(axis=1), 3))
print("Temperature-swap acceptance:",
      np.round(run.tswap_acceptance_fraction, 3))
print("Best log-likelihood per temperature:",
      np.round(np.nanmax(run.loglikelihood, axis=(1, 2)), 2))

In [ ]:
print("Current data rows:", len(data))
print("Gaia likelihood rows:", my_sys.gaia.n_obs)

In [ ]:
transit_errors = dict(zip(
    np.asarray(data["transit_id"], dtype=int),
    np.asarray(data["centroid_pos_error_al"], dtype=float),
))

with plt.style.context("default"):
    fig, ax = plt.subplots(figsize=(13, 4))

    for j, ccd in enumerate(ccd_data.group_by("transit_id").groups):
        transit_id = int(ccd["transit_id"][0])
        positions = np.asarray(ccd["centroid_pos_al"], dtype=float)
        errors = np.asarray(ccd["ipd_error_al"], dtype=float)
        x = j + 0.06 * (np.arange(len(ccd)) - (len(ccd) - 1) / 2)

        ax.errorbar(
            x,
            positions - np.median(positions),
            yerr=errors,
            fmt=".",
            color="tab:blue",
            alpha=0.6,
            label="individual CCDs" if j == 0 else None,
        )
        ax.errorbar(
            j,
            0,
            yerr=transit_errors[transit_id],
            fmt="o",
            color="crimson",
            capsize=2,
            label="combined transit" if j == 0 else None,
        )

    ax.axhline(0, color="0.5", linewidth=1)
    ax.set(
        xlabel="Transit index",
        ylabel="CCD AL ﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿﷿ transit median [mas]",
        ylim=(-1, 1),
    )
    ax.legend()

In [ ]:
# Compact joint fit summary: posterior, sky orbit, and AL residuals.
from matplotlib.collections import LineCollection
from matplotlib.lines import Line2D
from orbitize.plot import _get_star_offsets_dr4, _model_along_scan_dr4

joint_post = np.asarray(mcmc.results.post)
joint_lnlike = np.asarray(mcmc.results.lnlike)
joint_lnprior = np.zeros(len(joint_post))
for i, prior in enumerate(my_sys.sys_priors):
    if hasattr(prior, "compute_lnprob"):
        joint_lnprior += prior.compute_lnprob(joint_post[:, i])
best = joint_post[np.nanargmax(joint_lnlike + joint_lnprior)]

# Downsample before deriving masses, only for plotting.
corner_draws = rng.choice(
    len(joint_post), size=min(20_000, len(joint_post)), replace=False,
)
joint_plot_post = joint_post[corner_draws]
joint_standard = my_sys.basis.to_standard_basis(joint_plot_post.T.copy()).T
standard_idx = my_sys.basis.standard_basis_idx
period_samples = 365.25 * joint_plot_post[:, idx["per1"]]
summary_samples = np.column_stack([
    joint_plot_post[:, idx["m0"]],
    period_samples,
    joint_plot_post[:, idx["ecc1"]],
    np.rad2deg(joint_plot_post[:, idx["inc1"]]),
    joint_standard[:, standard_idx["m1"]] * u.Msun.to(u.Mjup),
    (np.rad2deg(joint_plot_post[:, idx["aop1"]]) + 180) % 360,
    np.rad2deg(joint_plot_post[:, idx["pan1"]]) % 360,
    joint_plot_post[:, idx["dr4_jitter"]],
])
summary_labels = [
    r"$M_0$ [$M_\odot$]", "P [day]", "e", "i [deg]",
    r"$M_1$ [$M_{\rm Jup}$]", r"$\omega_\star$ [deg]",
    r"$\Omega$ [deg]", "jitter [mas]",
]

published_summary = [
    stellar_mass, 83.9151, 0.3354, 6.2,
    10.98 / np.sin(np.deg2rad(6.2)), 201.28, None, None,
]

with plt.style.context("default"):
    # Render the corner plot once, then place it in the composite figure.
    corner_panel = corner.corner(
        summary_samples, labels=summary_labels,
        color="tab:blue", bins=30, smooth=1.0,
        quantiles=[0.16, 0.5, 0.84], show_titles=True,
        plot_datapoints=False, fill_contours=True,
        label_kwargs={"fontsize": 8},
        title_kwargs={"fontsize": 7},
    )
    corner.overplot_lines(
        corner_panel, published_summary, color="tab:red",
        linestyle="--", linewidth=1.0,
    )
    corner_panel.canvas.draw()
    corner_image = np.asarray(corner_panel.canvas.buffer_rgba()).copy()
    plt.close(corner_panel)

    fig = plt.figure(figsize=(20, 10))
    outer = fig.add_gridspec(
        2, 2, width_ratios=(1.35, 1), height_ratios=(1, 0.62),
        wspace=0.08, hspace=0.22,
    )

    ax_corner = fig.add_subplot(outer[:, 0])
    ax_corner.imshow(corner_image)
    ax_corner.axis("off")

    # MAP stellar reflex orbit using the main plot's coordinate helpers
    ax_sky = fig.add_subplot(outer[0, 1])
    best_period = 365.25 * best[idx["per1"]]
    orbit_epochs = np.linspace(ref_epoch_mjd, ref_epoch_mjd + best_period, 1_000)
    ra_orbit, dec_orbit = _get_star_offsets_dr4(
        best, my_sys, orbit_epochs
    )

    ra_epoch, dec_epoch = _get_star_offsets_dr4(
        best, my_sys, dr4.epochs_mjd
    )
    model_al = _model_along_scan_dr4(best, my_sys, dr4)
    al_residual = dr4.centroid_pos_al - model_al
    effective_sigma = np.hypot(
        dr4.centroid_pos_error_al, best[idx["dr4_jitter"]]
    )

    scan_unit = np.column_stack((dr4.sin_scan, dr4.cos_scan))
    anchor = np.column_stack((ra_epoch, dec_epoch))
    al_points = anchor + al_residual[:, None] * scan_unit
    al_errors = effective_sigma[:, None] * scan_unit
    al_segments = np.stack((al_points - al_errors, al_points + al_errors), axis=1)

    ax_sky.plot(ra_orbit, dec_orbit, color="#FF69AF", lw=1.8)
    ax_sky.add_collection(
        LineCollection(al_segments, colors="0.25", linewidths=0.8)
    )
    ax_sky.scatter(
        al_points[:, 0], al_points[:, 1], marker="*",
        s=80, color="k", edgecolor="0.2", linewidths=0.6, zorder=5,
    )
    ax_sky.plot(0, 0, "+", color="k", ms=12, mew=2)
    ax_sky.set(
        xlabel=r"$\Delta\alpha\cos\delta$ [mas]",
        ylabel=r"$\Delta\delta$ [mas]",
        title="MAP joint RV + Gaia DR4 orbit",
    )
    ax_sky.set_aspect("equal", adjustable="datalim")
    ax_sky.invert_xaxis()
    ax_sky.axhline(0, color="0.85", ls="--", lw=0.8)
    ax_sky.axvline(0, color="0.85", ls="--", lw=0.8)

    # Along-scan data/model comparison and normalized residual histogram.
    time_grid = outer[1, 1].subgridspec(
        2, 2, height_ratios=(1.2, 0.8), width_ratios=(5, 1),
        hspace=0.05, wspace=0.05,
    )
    ax_al = fig.add_subplot(time_grid[0, :])
    ax_resid = fig.add_subplot(time_grid[1, 0], sharex=ax_al)
    ax_hist = fig.add_subplot(time_grid[1, 1], sharey=ax_resid)

    ax_al.errorbar(
        dr4.epochs_mjd, dr4.centroid_pos_al, yerr=effective_sigma,
        fmt="o", ms=3, mfc="white", mec="k", ecolor="0.6",
        label="Gaia DR4",
    )
    ax_al.plot(dr4.epochs_mjd, model_al, ".", color="tab:blue", label="model")
    ax_al.set_ylabel("AL abscissa [mas]")
    ax_al.legend(frameon=False, fontsize=8, ncol=2)
    ax_al.tick_params(labelbottom=False)

    normalized_residual = al_residual / effective_sigma
    residual_limit = max(4, np.ceil(np.max(np.abs(normalized_residual))))
    ax_resid.errorbar(
        dr4.epochs_mjd, normalized_residual, yerr=np.ones(len(data)),
        fmt=".", color="0.2", ecolor="0.65", ms=3,
    )
    ax_resid.axhline(0, color="tab:blue", lw=1)
    ax_resid.set(
        xlabel="Epoch [MJD]", ylabel=r"$(\mathrm{data-model})/\sigma$",
        ylim=(-residual_limit, residual_limit),
    )

    residual_bins = np.linspace(-residual_limit, residual_limit, 25)
    ax_hist.hist(
        normalized_residual, bins=residual_bins, density=True,
        orientation="horizontal", histtype="step", color="k",
    )
    residual_y = np.linspace(-residual_limit, residual_limit, 300)
    unit_gaussian = np.exp(-0.5 * residual_y**2) / np.sqrt(2 * np.pi)
    ax_hist.plot(unit_gaussian, residual_y, color="tab:blue")
    ax_hist.tick_params(labelleft=False)
    ax_hist.set_xlabel("density", fontsize=8)

    fig.legend(
        handles=[
            Line2D([0], [0], color="tab:blue", lw=2, label="Joint RV + Gaia DR4"),
            Line2D([0], [0], color="tab:red", ls="--", label="Published comparison"),
        ],
        loc="upper right", bbox_to_anchor=(0.98, 0.985),
        ncol=2, frameon=False,
    )
    fig.suptitle("HD 114762 B", fontsize=20, y=0.985)

In [ ]:
# RV-only summary: posterior, phase-folded orbit, and RV residuals.
from matplotlib.lines import Line2D

rv_samples = np.asarray(rv_mcmc.results.post)
rv_lnlikes = np.asarray(rv_mcmc.results.lnlike)
rv_best = rv_samples[np.nanargmax(rv_lnlikes)]

# Convert only a plotting subset to obtain the derived companion mass.
rv_plot_draws = rng.choice(
    len(rv_samples), size=min(20_000, len(rv_samples)), replace=False,
)
rv_plot_samples = rv_samples[rv_plot_draws]
rv_standard_plot = rv_system.basis.to_standard_basis(
    rv_plot_samples.T.copy()
).T
rv_standard_idx = rv_system.basis.standard_basis_idx
rv_m1_sini = (
    rv_standard_plot[:, rv_standard_idx["m1"]]
    * np.sin(rv_standard_plot[:, rv_standard_idx["inc1"]])
    * u.Msun.to(u.Mjup)
)
rv_summary_samples = np.column_stack([
    365.25 * rv_plot_samples[:, rv_idx["per1"]],
    rv_plot_samples[:, rv_idx["ecc1"]],
    rv_plot_samples[:, rv_idx["K1"]],
    (np.rad2deg(rv_plot_samples[:, rv_idx["aop1"]]) + 180) % 360,
    rv_plot_samples[:, rv_idx["tau1"]],
    rv_m1_sini,
    rv_plot_samples[:, rv_idx["m0"]],
])
rv_summary_labels = [
    "P [day]", "e", "K [km/s]", r"$\omega_\star$ [deg]",
    r"$\tau$", r"$M_1\sin i$ [$M_{\rm Jup}$]",
    r"$M_0$ [$M_\odot$]",
]
published_tau = (
    (2449889.106 - 2400000.5 - ref_epoch_mjd) / 83.9151
) % 1
published_rv_summary = [
    83.9151, 0.3354, 0.61248, 201.28, published_tau, 10.98, stellar_mass,
]

rv_model, rv_jitter = rv_system.compute_model(rv_best)
rv_model = np.asarray(rv_model[:, 0], dtype=float)
rv_effective_sigma = np.hypot(
    np.asarray(rv_data["quant1_err"], dtype=float),
    np.asarray(rv_jitter[:, 0], dtype=float),
)
rv_values = np.asarray(rv_data["quant1"], dtype=float)
rv_epochs = np.asarray(rv_data["epoch"], dtype=float)
rv_residual = rv_values - rv_model
rv_normalized_residual = rv_residual / rv_effective_sigma
rv_instruments = np.asarray(rv_data["instrument"]).astype(str)
unique_rv_instruments = np.unique(rv_instruments)
instrument_colors = plt.cm.tab10(
    np.linspace(0, 1, len(unique_rv_instruments))
)

# Remove the fitted detector zero points for the phase-folded panel.
rv_centered = rv_values.copy()
for instrument in unique_rv_instruments:
    use = rv_instruments == instrument
    rv_centered[use] -= rv_best[rv_idx[f"gamma_{instrument}"]]

rv_best_period_days = 365.25 * rv_best[rv_idx["per1"]]
rv_phase = (
    (rv_epochs - ref_epoch_mjd) / rv_best_period_days
    - rv_best[rv_idx["tau1"]]
) % 1
phase_grid = np.linspace(0, 1, 1_000)
phase_epochs = ref_epoch_mjd + (rv_best[rv_idx["tau1"]] + phase_grid) * rv_best_period_days
rv_best_standard = rv_system.basis.to_standard_basis(rv_best.copy())
_, _, phase_velocity, _ = rv_system.compute_all_orbits(
    rv_best_standard, epochs=phase_epochs
)
phase_velocity = phase_velocity[:, 0, 0]

with plt.style.context("default"):
    rv_corner_panel = corner.corner(
        rv_summary_samples, labels=rv_summary_labels,
        color="tab:blue", bins=30, smooth=1.0,
        quantiles=[0.16, 0.5, 0.84], show_titles=True,
        plot_datapoints=False, fill_contours=True,
        label_kwargs={"fontsize": 8},
        title_kwargs={"fontsize": 7},
    )
    corner.overplot_lines(
        rv_corner_panel, published_rv_summary, color="tab:red",
        linestyle="--", linewidth=1.0,
    )
    rv_corner_panel.canvas.draw()
    rv_corner_image = np.asarray(
        rv_corner_panel.canvas.buffer_rgba()
    ).copy()
    plt.close(rv_corner_panel)

    fig = plt.figure(figsize=(20, 10))
    outer = fig.add_gridspec(
        2, 2, width_ratios=(1.35, 1), height_ratios=(1, 0.62),
        wspace=0.08, hspace=0.22,
    )

    ax_corner = fig.add_subplot(outer[:, 0])
    ax_corner.imshow(rv_corner_image)
    ax_corner.axis("off")

    ax_phase = fig.add_subplot(outer[0, 1])
    for instrument, color in zip(unique_rv_instruments, instrument_colors):
        use = rv_instruments == instrument
        ax_phase.errorbar(
            rv_phase[use], 1_000 * rv_centered[use],
            yerr=1_000 * rv_effective_sigma[use], fmt="o", ms=4,
            color=color, label=instrument,
        )
    ax_phase.plot(phase_grid, 1_000 * phase_velocity, color="k", lw=1.5)
    ax_phase.set(
        xlabel="Orbital phase from periastron",
        ylabel="Offset-corrected RV [m/s]",
        title="Best RV-only Keplerian orbit", xlim=(0, 1),
    )
    ax_phase.legend(frameon=False, fontsize=7, ncol=2)

    time_grid = outer[1, 1].subgridspec(
        2, 2, height_ratios=(1.2, 0.8), width_ratios=(5, 1),
        hspace=0.05, wspace=0.05,
    )
    ax_rv = fig.add_subplot(time_grid[0, :])
    ax_resid = fig.add_subplot(time_grid[1, 0], sharex=ax_rv)
    ax_hist = fig.add_subplot(time_grid[1, 1], sharey=ax_resid)

    for instrument, color in zip(unique_rv_instruments, instrument_colors):
        use = rv_instruments == instrument
        ax_rv.errorbar(
            rv_epochs[use], 1_000 * rv_values[use],
            yerr=1_000 * rv_effective_sigma[use], fmt="o", ms=3,
            color=color,
        )
        ax_rv.plot(
            rv_epochs[use], 1_000 * rv_model[use], ".",
            color="k", ms=3,
        )
    ax_rv.set_ylabel("Relative RV [m/s]")
    ax_rv.tick_params(labelbottom=False)

    ax_resid.errorbar(
        rv_epochs, rv_normalized_residual,
        yerr=np.ones(len(rv_epochs)), fmt=".",
        color="0.2", ecolor="0.65", ms=3,
    )
    ax_resid.axhline(0, color="tab:blue", lw=1)
    rv_residual_limit = max(4, np.ceil(np.max(np.abs(rv_normalized_residual))))
    ax_resid.set(
        xlabel="Epoch [MJD]", ylabel=r"$(\mathrm{data-model})/\sigma$",
        ylim=(-rv_residual_limit, rv_residual_limit),
    )

    rv_residual_bins = np.linspace(-rv_residual_limit, rv_residual_limit, 25)
    ax_hist.hist(
        rv_normalized_residual, bins=rv_residual_bins, density=True,
        orientation="horizontal", histtype="step", color="k",
    )
    rv_residual_y = np.linspace(-rv_residual_limit, rv_residual_limit, 300)
    rv_unit_gaussian = np.exp(-0.5 * rv_residual_y**2) / np.sqrt(2 * np.pi)
    ax_hist.plot(rv_unit_gaussian, rv_residual_y, color="tab:blue")
    ax_hist.tick_params(labelleft=False)
    ax_hist.set_xlabel("density", fontsize=8)

    fig.legend(
        handles=[
            Line2D([0], [0], color="tab:blue", lw=2, label="RV-only posterior"),
            Line2D([0], [0], color="tab:red", ls="--", label="Kane et al. (2011)"),
        ],
        loc="upper right", bbox_to_anchor=(0.98, 0.985),
        ncol=2, frameon=False,
    )
    fig.suptitle("HD 114762 B: RV-only fit", fontsize=20, y=0.985)